# 03 - Phase 3: Frozen DINOv2 Foundation Backbone

**One-liner:** drop a frozen DINOv2 ViT-B/14 in as the backbone, train only the synthetic pyramid + detector neck/heads + STTran graph head. This is the result that beats the original STTran paper.

Phase 3 is *not* a rewrite of Phase 2. It is a parallel branch that reuses the corrected Phase 2 detector shell + STTran heads but adds DINOv2-specific files for the backbone and preprocessing. The whole point of keeping it parallel was to avoid contaminating the working Phase 2 code while we experimented.

## The Phase 3 architecture

```mermaid
flowchart LR
    Frame["AG frame, 1024 input, LSJ [0.5, 2.0]"] --> DINOv2["DINOv2 ViT-B/14 (FROZEN)"]
    DINOv2 --> Tokens["patch tokens"]
    Tokens --> Bridge["DINOv2Bridge: 1x1 proj + p2/p3/p4/p5"]
    Bridge --> RPN[RPN]
    RPN --> RoI[RoI Align]
    RoI --> Heads["RCNN heads (trainable)"]
    Heads --> STTran[STTran heads]
    STTran --> Out["per-frame scene graph"]
```

The flow is identical to Phase 2 conceptually. Three concrete differences:
1. The backbone is **DINOv2 ViT-B/14** instead of plain ViT-B/16 (note the patch size: 14, not 16).
2. The backbone is **frozen** (`requires_grad=False` on every param).
3. The bridge enforces **DINOv2-native photometric preprocessing** as a single source of truth.

**Repo files for this phase:**
- `phase3_dinov2/dinov2_bridge.py` - the frozen DINOv2 bridge module
- `phase3_dinov2/preprocess.py` - the single-source-of-truth preprocessing config
- `STTran/lib/object_detector_phase3.py` - thin wrapper that asserts the DINOv2 path is in use
- `STTran/dataloader/action_genome_phase3.py` - graph-data loader using DINOv2 preprocessing
- `STTran/dataloader/action_genome_detector_phase3.py` - detector-data loader using DINOv2 preprocessing
- `STTran/train_detector_stage1_phase3.py` - Stage 1 launcher
- `STTran/train_phase3.py` - Stage 2 launcher
- `STTran/evaluate_only_phase3.py` - eval launcher

## Why a *separate* parallel branch?

From `Big Picture.pdf`:

> Phase 3 is intentionally separate. We did not rewrite the working Phase 2 path. Instead we created a new DINOv2 branch with its own files so we can experiment safely.

Three reasons this matters:

1. **Phase 2 had just been stabilized** after three bugs. We didn't want one stray DINOv2 change to silently break the corrected ViT baseline that we needed for the comparison table.
2. **DINOv2 has different preprocessing requirements** than `vit_base_patch16_224` (different mean/std, patch size, interpolation). Mixing them in one branch invites subtle Phase-2-style mixed-path bugs.
3. **The frozen-vs-trainable code paths are different**: frozen needs `with torch.no_grad():` around the backbone forward and `backbone.eval()` permanently, which would complicate a unified path. Cleaner to fork.

**Asked-about gotcha:** even though Phase 3 is parallel, it *reuses the same `STTran/lib/sttran.py`* (and the same SGCLS duplicate-policy fix). So the corrected SGCLS behaviour transferred for free.

## DINOv2Bridge - the frozen backbone module

Lives in `phase3_dinov2/dinov2_bridge.py`. Same idea as the Phase 2 ViTDet bridge, with three additions: model-native preprocessing, frozen-mode permanent eval, and explicit per-mode runtime logging.

### Loading DINOv2 from `timm`
Lines 65-71:

```python
self.backbone = timm.create_model(
    self.model_name,                 # 'vit_base_patch14_dinov2'
    pretrained=True,                 # downloads Meta's DINOv2 weights via timm
    num_classes=0,                   # we don't want the classification head
    dynamic_img_size=cfg.dynamic_img_size,  # support 1024 input even though training was 224
    dynamic_img_pad=cfg.dynamic_img_pad,
)
```

### Synthetic pyramid (same shape as Phase 2)
Lines 92-109:

```python
self.p2 = nn.Sequential(
    nn.ConvTranspose2d(self.embed_dim, self.out_channels, kernel_size=2, stride=2),
    nn.GELU(),
    nn.Conv2d(self.out_channels, self.out_channels, kernel_size=3, padding=1),
    nn.GELU(),
)
self.p3 = nn.Sequential(nn.Conv2d(self.embed_dim, self.out_channels, kernel_size=1), nn.GELU())
self.p4 = nn.Sequential(nn.Conv2d(self.embed_dim, self.out_channels, kernel_size=3, stride=2, padding=1), nn.GELU())
self.p5 = nn.Sequential(nn.Conv2d(self.out_channels, self.out_channels, kernel_size=3, stride=2, padding=1), nn.GELU())
```

Strides: 1/4, 1/8, 1/16, 1/32 - same family as ViTDet.

### Freezing the backbone (the actual `requires_grad=False`)
Lines 120-123:

```python
if self.freeze_backbone:
    for param in self.backbone.parameters():
        param.requires_grad_(False)
    self.backbone.eval()
```

And the `train()` override (lines 130-134) keeps the backbone in eval mode forever, even when the rest of the module is set to `train()`:

```python
def train(self, mode: bool = True):
    super().train(mode)
    if self.freeze_backbone:
        self.backbone.eval()
    return self
```

**Why this matters:** if you forgot the `train()` override, calling `model.train()` would flip the backbone's BN/dropout etc. into training mode and your "frozen" backbone would silently behave differently between train and eval calls. This is the kind of bug that would have wasted us another two days.

### The forward pass with `torch.no_grad()`
Lines 229-237:

```python
def forward(self, images: Tensor) -> Dict[str, Tensor]:
    normalized = self._normalize_images(images)
    height, width = int(normalized.shape[2]), int(normalized.shape[3])
    self._input_hw = (height, width)
    if self.freeze_backbone:
        with torch.no_grad():
            features = self.backbone.forward_features(normalized)
    else:
        features = self.backbone.forward_features(normalized)
```

When frozen, the backbone forward is wrapped in `no_grad()` - we don't even build a computation graph through DINOv2. Dramatic memory savings on H100, since 86M backbone params don't need activation memory.

### Mathematical sketch of the frozen path

Let $f_\theta$ be DINOv2 with frozen weights $\theta$:
$$\nabla_\theta f_\theta \equiv 0$$

We extract patch tokens, drop the prefix tokens (CLS / register tokens), reshape into a 2D grid, and apply the synthetic pyramid:
$$\tilde{F} = \frac{f_\theta(I_t)}{\|f_\theta(I_t)\|_2}$$
$$F_s = U_s(\tilde{F}),\ s \in \{1/4, 1/8, 1/16, 1/32\}$$

Only the parameters in $\{U_s\}$, the align layers, the RPN, the detector heads, and the STTran graph head receive gradient updates. **The total trainable parameter count drops by roughly 86M** (the size of ViT-B), which is why the run actually fits in memory at batch size 12 (vs. batch size 8 for the trainable Phase 2 ViT).

## Single source of truth for preprocessing

The Mixed-Path Bug from Phase 2 was a stark lesson: any time two code paths can diverge silently, they will. Phase 3 attacks that lesson head-on.

All photometric preprocessing for Phase 3 is in *one* file: `phase3_dinov2/preprocess.py`. The dataloaders, the bridge, and the eval launchers all read from the same `DINOv2VisualConfig` dataclass.

From `phase3_dinov2/preprocess.py` lines 11-26:

```python
@dataclass(frozen=True)
class DINOv2VisualConfig:
    model_name: str
    input_size: int
    train_lsj_min: float
    train_lsj_max: float
    patch_size: int
    detector_stride: int
    interpolation_name: str
    mean: Tuple[float, float, float]
    std: Tuple[float, float, float]
    pad_rgb: Tuple[float, float, float]
    dynamic_img_size: bool
    dynamic_img_pad: bool
    freeze_backbone: bool
    override_summary: str
```

And the construction reads `mean`, `std`, `interpolation` from `timm.resolve_data_config(...)` - i.e. **whatever values DINOv2 was originally trained with**, not whatever defaults Faster R-CNN happens to use. Lines 94-97:

```python
data_cfg, patch_size = _resolve_base_timm_cfg(model_name)
mean = tuple(float(x) for x in data_cfg.get('mean', (0.485, 0.456, 0.406)))
std = tuple(float(x) for x in data_cfg.get('std', (0.229, 0.224, 0.225)))
interpolation_name = interpolation_name or str(data_cfg.get('interpolation', 'bicubic'))
```

**Translation in plain English:** the bridge normalizes images using DINOv2's own mean/std, not the ImageNet defaults. If you feed DINOv2 ImageNet-mean-normalized images, its features quietly degrade. We don't do that.

The same config is read by:
- `phase3_dinov2/dinov2_bridge.py:33` (in the bridge `__init__`)
- `STTran/dataloader/action_genome_phase3.py:100` (graph-data loader)
- `STTran/dataloader/action_genome_detector_phase3.py:164` (detector-data loader)

## Anti-regression safeguards (so we don't repeat Phase 2's mistakes)

From `Big Picture.pdf`:

> We built explicit safeguards so Phase 3 does not repeat the early ViT mistakes:
> - one preprocessing config shared across train/eval
> - one explicit DINO backbone bridge
> - path assertions through PHASE3_ASSERT_DINOV2_PATH
> - mode-specific runtime logging of: backbone name, bridge class, feature shape, preprocessing config, label source
> - SGCLS defaults locked in the Phase 3 wrappers: SGCLS_DUPLICATE_POLICY=iou and SGCLS_LABEL_SOURCE=detector

### Per-mode runtime logging in code
`phase3_dinov2/dinov2_bridge.py` lines 212-227:

```python
def _log_runtime_once(self, base: Tensor):
    mode_tag = self._runtime_context.get('mode_tag', 'unset')
    label_source = self._runtime_context.get('label_source', 'unset')
    if mode_tag in self._runtime_logged_modes:
        return
    self._runtime_logged_modes.add(mode_tag)
    print(
        'Phase3 backbone path [{}]: backbone=dinov2 bridge={} feature_shape={} preprocessing={} frozen={} label_source={}'.format(
            mode_tag,
            self.__class__.__name__,
            tuple(base.shape),
            format_visual_config(self.visual_cfg),
            self.freeze_backbone,
            label_source,
        )
    )
```

The first time PredCLS calls `forward()`, this prints `Phase3 backbone path [predcls]: backbone=dinov2 bridge=DINOv2Bridge ...`. Same for SGCLS and SGDET. If any mode is silently using the wrong path, the log will tell you in epoch 0.

### Assertion in the wrapper
`STTran/lib/object_detector_phase3.py` (around line 56) asserts the DINOv2 path is actually being used at runtime when `PHASE3_ASSERT_DINOV2_PATH=1`. The launchers default this to 1 (`STTran/scripts/train_phase3_dinov2_h100.sbatch:115`).

### SGCLS defaults locked in
Every Phase 3 launch script sets:
```bash
export SGCLS_DUPLICATE_POLICY="${SGCLS_DUPLICATE_POLICY:-iou}"
export SGCLS_LABEL_SOURCE="${SGCLS_LABEL_SOURCE:-detector}"
```
(`STTran/scripts/train_phase3_dinov2_h100.sbatch:116-117`).

So the SGCLS fix from Phase 2 is now part of the Phase 3 launcher contract.

## Why DINOv2 wins (the explanation your partner sent)

Verbatim from the partner's message, unpacked into bullet form so you can present it:

**Reason 1 - DINOv2 is a *general-purpose foundation model***
> Although both backbones are transformer-based, DINOv2 behaves more like a general-purpose foundation model: its frozen features appear to preserve object semantics, spatial structure, and long-tail category information more effectively

Translation: DINOv2 was trained on a curated 142M-image self-supervised corpus. Action Genome has *thousands* of training videos. A from-scratch ViT learning detector features only sees AG; DINOv2 has already seen a vast world of objects.

**Reason 2 - frozen features improve the harder end-to-end tasks**
> This is reflected in our results, where DINOv2 substantially improves SGCLS and SGDET compared with the ViT branch, even when detector-only mAP differences are modest.

Translation: detector mAP is essentially the same (23.19 ViT vs 24.25 DINOv2), but SGCLS jumps by 22 absolute points. This is the strongest single piece of evidence that what changes is the *embedding quality per box*, not the localization.

**Reason 3 - we inherited a corrected pipeline**
> the DINOv2 branch benefited from the corrected pipeline we established after debugging the ViT integration, including consistent backbone paths across PredCLS/SGCLS/SGDET, model-native preprocessing, and safer SGCLS evaluation logic.

Translation: Phase 3 didn't have to discover the three Phase 2 bugs. It started clean.

**Bottom line:**
> DINOv2 is not simply "another ViT" in this project; it serves as a stronger frozen feature extractor whose representations are better aligned with downstream object reasoning and relationship prediction.

## Phase 3 final scorecard

From the final report and `sttran_backbone_migration_report_latest.pdf`:

| Metric | Base STTran | Phase 2 ViT | **Phase 3 DINOv2** | Win? |
|---|---|---|---|---|
| Detector mAP@0.5 | 24.6 | 23.19 | **24.25** | tie / very close to base |
| PredCLS R@20 (With) | 71.8 | 44.95 | **71.71** | tie |
| **SGCLS R@20 (With)** | 47.5 | 30.75 | **52.88** | **+5.4 over base, +22.1 over ViT** |
| **SGDET R@20 (With)** | 34.1 | 23.07 | **40.41** | **+6.3 over base, +17.3 over ViT** |

Headline: **SGCLS R@20 = 52.88 vs base paper 47.5**, **SGDET R@20 = 40.41 vs base paper 34.1**.

The two metrics where DINOv2 is *not* clearly winning are PredCLS (where there is no embedding work to do - boxes and classes are given, only relations are predicted, so backbone choice doesn't matter much) and detector mAP (where DINOv2 is essentially tied with base ResNet, since detection is dominated by RPN/RoI quality).

## A subtler Phase 3 lesson: best detector mAP =/= best graph downstream

From `sttran_backbone_migration_report_latest.pdf`:

> A second important lesson in Phase 3 was that the detector checkpoint with the highest standalone mAP was not automatically the best detector for Stage 2. To solve this, detector checkpoints were ranked not only by detector mAP, but also by short Stage-2 downstream performance, especially with-constraint SGDET R@20.

**Why:** SGCLS/SGDET use *predicted* class scores at inference. A detector that picks classes well, even at slightly lower box-AP, can produce better graph metrics than one that has higher mAP but worse class confidence. So we ran short Stage-2 sweeps over candidate Stage-1 checkpoints and ranked them by SGDET R@20.

Code for this exists in `STTran/scripts/collect_phase3_rank_results.py` and `STTran/scripts/submit_phase3_dinov2_rank_sweep.sh`.

**Talking-point version:** *"We learned the hard way that the highest detector-mAP checkpoint isn't always the best one for the downstream graph task, so we added a tiny Stage-2 ranking sweep on top of Stage-1 to pick the right checkpoint."*

## How to talk about Phase 3 in 60 seconds

> "Phase 3 swapped the trainable ViT for a *frozen* DINOv2 ViT-B/14 backbone. We made it a parallel branch instead of editing Phase 2 so we could experiment safely without breaking the corrected ViT scaffolding. The key file is `phase3_dinov2/dinov2_bridge.py`, which freezes every backbone parameter, runs the backbone forward inside `torch.no_grad()` to save memory, and synthesizes the same four-level pyramid as Phase 2. We added three anti-regression safeguards: a single source of truth for preprocessing in `phase3_dinov2/preprocess.py` so train and eval cannot diverge, runtime path assertions that the DINOv2 path is actually being used, and per-mode logging that prints the backbone, bridge, and feature shape the first time each mode is called. Result: SGCLS R@20 = 52.88, SGDET R@20 = 40.41 - both *over* the original STTran paper's published numbers - while training only a tiny fraction of the parameters. DINOv2's strength is that it preserves fine-grained object semantics and spatial structure from its 142M-image self-supervised pretraining, which is exactly what relation prediction needs."

Practice this. It is the sentence that closes the presentation.

---
**Next:** open `04_results_and_story.ipynb` for the tables and the four insights to recite.